# Telemetrix query tour

Read stored telemetry with `telemetrix-query`: Tag metadata, an Asset's Measurements over an interval, Quality filters and one-minute Rollups.

Before running:

1. `pip install ./sdk/python` (from the repository root) in this notebook's environment.
2. A reader login exists (see `sdk/python/README.md`, *Database access*).
3. `TELEMETRIX_DATABASE_URL` and the password source (`PGPASSWORD` or `~/.pgpass`) are set in the environment that started Jupyter.

In [ ]:
# Parameters: the Asset and a timezone-aware interval (start <= time < end).
from datetime import datetime, timedelta, timezone

ASSET = "Pump01"
END = datetime.now(timezone.utc)
START = END - timedelta(hours=1)

In [ ]:
from telemetrix_query import Quality, connect

client = connect()  # read-only session; raises ConnectionFailedError with a hint if settings are wrong
client.capabilities

## Tag metadata

In [ ]:
tags = client.tags(asset_id=ASSET)
for tag in tags:
    print(f"{tag.tag_id:>3} {tag.name:<20} unit={tag.unit} range=[{tag.min_valid}, {tag.max_valid}] derived_by={tag.derived_by}")

## Measurements for the Asset and interval

A page holds at most `limit` Measurements ordered by time and Tag; `page.next_cursor` continues it. Every time is UTC.

In [ ]:
page = client.measurements(START, END, asset_id=ASSET, limit=10)
for m in page:
    print(m.time.isoformat(), m.tag, m.value, m.unit, m.quality.name)
print("more pages:", page.has_more)

## Quality filters

`iter_measurements` streams every match one page at a time. A null value (`None`) is not the same as Bad Quality: a Bad Measurement may still carry the out-of-range value that made it Bad.

In [ ]:
bad = list(client.iter_measurements(START, END, asset_id=ASSET, qualities=Quality.BAD))
null_values = sum(m.value is None for m in bad)
print(f"{len(bad)} Bad Measurements, {null_values} of them without a value")

## One-minute Rollups

Rollups summarise Good Measurements only, per Tag and minute. The newest minutes appear after the continuous aggregate refreshes; query Measurements for them.

In [ ]:
rollups = list(client.iter_rollups(START, END, asset_id=ASSET))
for r in rollups[:10]:
    print(r.bucket.isoformat(), r.tag, f"min={r.min_value} max={r.max_value} avg={r.avg_value} n={r.sample_count}")
print(len(rollups), "rollups")

## Optional: pandas

The SDK does not depend on pandas; its results are dataclasses, so they convert directly.

In [ ]:
from dataclasses import asdict

try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas is not installed; skipping the DataFrame")
if pd is not None:
    frame = pd.DataFrame([asdict(r) for r in rollups])
    display(frame.pivot(index="bucket", columns="tag", values="avg_value").head())

In [ ]:
client.close()